> **Deprecated**
>
> This notebook has been consolidated into `secondary_documentation_notebook.ipynb`.
> Please refer to that notebook instead.
> 
> Section: **Downloading DEMs for Representative Areas** (Section 5 in `secondary_documentation_notebook.ipynb`).

In [1]:
import os
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.errors import RasterioIOError
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.merge import merge as rio_merge
import pickle
from pyproj import CRS
import io


In [2]:
with open('output/tabular_and_text/representative_positions.pkl', 'rb') as f:
    positions_ordered = pickle.load(f)

In [3]:


# GDAL: HTTP-Range-Requests auf COGs beschleunigen
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif"


def km_to_degrees(km, lat):
    """Rechnet km in Grad um. Returns (delta_lon, delta_lat)."""
    km_per_degree_lat = 111.0
    km_per_degree_lon = 111.0 * np.cos(np.radians(lat))
    return km / km_per_degree_lon, km / km_per_degree_lat


def cop30_tile_url(lon_int, lat_int):
    """URL einer 1°×1°-COP30-Tile im AWS-Bucket von Sinergise."""
    ns = "N" if lat_int >= 0 else "S"
    ew = "E" if lon_int >= 0 else "W"
    name = (
        f"Copernicus_DSM_COG_10_"
        f"{ns}{abs(lat_int):02d}_00_"
        f"{ew}{abs(lon_int):03d}_00_DEM"
    )
    return f"https://copernicus-dem-30m.s3.amazonaws.com/{name}/{name}.tif"


def download_dem(lon, lat, extent_km, output_folder, label, idx):
    """Lädt ein DEM-GeoTIFF für eine Position."""
    delta_lon, delta_lat = km_to_degrees(extent_km / 2, lat)
    west, east = lon - delta_lon, lon + delta_lon
    south, north = lat - delta_lat, lat + delta_lat
    
    filename = f"{label}_{idx}_dem_{lon:.4f}_{lat:.4f}.tif"
    filepath = os.path.join(output_folder, filename)
    
    if os.path.exists(filepath):
        print(f"Überspringe {filename} (existiert bereits)")
        return
    
    # Welche 1°×1°-Tiles berührt die Bounding Box?
    lon_tiles = range(int(np.floor(west)), int(np.floor(east)) + 1)
    lat_tiles = range(int(np.floor(south)), int(np.floor(north)) + 1)
    
    print(f"Lade {filename}...")
    
    datasets = []
    for lon_int in lon_tiles:
        for lat_int in lat_tiles:
            url = cop30_tile_url(lon_int, lat_int)
            try:
                # /vsicurl/ holt per HTTP-Range-Request nur die nötigen Bytes
                datasets.append(rasterio.open(f"/vsicurl/{url}"))
            except RasterioIOError:
                # Tile fehlt (reiner Ozean, oder Land ohne öffentliche Freigabe)
                pass
    
    if not datasets:
        print(f"  ❌ Keine Tiles verfügbar für diesen Bereich")
        return
    
    # Tiles mergen und auf die gewünschte Bounding Box zuschneiden
    mosaic, out_transform = merge(datasets, bounds=(west, south, east, north))
    
    out_meta = datasets[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform,
        "compress": "deflate",
    })
    
    with rasterio.open(filepath, "w", **out_meta) as dst:
        dst.write(mosaic)
    
    for src in datasets:
        src.close()
    
    print(f"  Gespeichert: {filename}")


def download_representative_samples(
    positions_dict,
    extent_km=15,
    output_folder="Representative_Samples",
):
    """Lädt DEMs für alle Positionen."""
    os.makedirs(output_folder, exist_ok=True)
    
    for label, positions in positions_dict.items():
        for idx, position in enumerate(positions):
            lon, lat = float(position[0]), float(position[1])
            download_dem(lon, lat, extent_km, output_folder, label, idx)


def download_dem_aeqd(lon, lat, side_km, output_folder, label, idx):
    """
    Lädt ein DEM, projiziert es azimuthal-äquidistant um (lon, lat)
    und speichert einen quadratischen Ausschnitt der Größe side_km × side_km.
    """
    # 2× Puffer beim Download
    download_km = side_km * 2

    delta_lon, delta_lat = km_to_degrees(download_km / 2, lat)
    west, east = lon - delta_lon, lon + delta_lon
    south, north = lat - delta_lat, lat + delta_lat

    filename = f"{label}_{idx}_aeqd_{lon:.4f}_{lat:.4f}.tif"
    filepath = os.path.join(output_folder, filename)

    if os.path.exists(filepath):
        print(f"Überspringe {filename} (existiert bereits)")
        return

    # Tiles laden
    lon_tiles = range(int(np.floor(west)), int(np.floor(east)) + 1)
    lat_tiles = range(int(np.floor(south)), int(np.floor(north)) + 1)

    print(f"Lade {filename}...")

    datasets = []
    for lon_int in lon_tiles:
        for lat_int in lat_tiles:
            url = cop30_tile_url(lon_int, lat_int)
            try:
                datasets.append(rasterio.open(f"/vsicurl/{url}"))
            except RasterioIOError:
                pass

    if not datasets:
        print(f"  ❌ Keine Tiles verfügbar")
        return

    # Mosaic im WGS84-Bereich zusammensetzen
    mosaic, mosaic_transform = merge(datasets, bounds=(west, south, east, north))
    src_crs = datasets[0].crs
    for ds in datasets:
        ds.close()

    # Ziel-CRS: azimuthal äquidistant, zentriert auf (lon, lat)
    dst_crs = CRS.from_proj4(
        f"+proj=aeqd +lat_0={lat} +lon_0={lon} +datum=WGS84 +units=m"
    )

    print("Mosaic-Min:", np.min(mosaic))
    print("Mosaic-Max:", np.max(mosaic))
    print("Mosaic-Diff:", np.max(mosaic)-np.min(mosaic))

    # Auflösung aus dem Mosaic übernehmen (~30 m für COP30)
    res_m = mosaic_transform.a * 111_000  # grobe Umrechnung Grad → Meter

    # Ziel-Ausdehnung: nur der gewünschte Ausschnitt (kein Puffer mehr)
    half = (side_km * 1000) / 2
    dst_bounds = (-half, -half, half, half)  # (west, south, east, north) in Metern

    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs,
        dst_crs,
        mosaic.shape[2],
        mosaic.shape[1],
        left=west, bottom=south, right=east, top=north,
        dst_width=int(side_km * 1000 / res_m),
        dst_height=int(side_km * 1000 / res_m),
    )

    # Transform manuell auf den Ausschnitt setzen
    pixel_size = side_km * 1000 / int(side_km * 1000 / res_m)
    dst_transform = rasterio.transform.from_bounds(
        *dst_bounds,
        width=int(side_km * 1000 / pixel_size),
        height=int(side_km * 1000 / pixel_size),
    )
    dst_width = dst_height = int(side_km * 1000 / pixel_size)


    dst_data = np.zeros((1, dst_height, dst_width), dtype=np.uint16)



    reproject(
        source=mosaic,
        destination=dst_data,
        src_transform=mosaic_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
    )

    print("DST-Min:", np.min(dst_data))
    print("DST-Max:", np.max(dst_data))
    print("DST-Diff:", np.max(dst_data)-np.min(dst_data))

    
    out_meta = {
        "driver": "GTiff",
        "dtype": "uint16",
        "width": dst_width,
        "height": dst_height,
        "count": 1,
        "crs": dst_crs,
        "transform": dst_transform,
        "compress": "deflate",
    }

    with rasterio.open(filepath, "w", **out_meta) as dst:
        dst.write(dst_data)

    print(f"  Gespeichert: {filename} ({dst_width}×{dst_height} px)")



In [ ]:
documentation_italy_tile = (11.295510152613245, 42.910659637077316)

download_dem_aeqd(
    *documentation_italy_tile,
    12,
    "/Users/scharnagl/Documents/GitHub/geospatial-landscape-clustering-by-fft/documentation/Representative_Samples/16_bit_heightmaps",
    "italy",
    "sample_tile_16bit",
)

Lade italy_sample_tile_aeqd_11.2955_42.9107.tif...
Mosaic-Min: 18.5
Mosaic-Max: 642.4154
Mosaic-Diff: 623.9154
DST-Min: 29
DST-Max: 365
DST-Diff: 336
  Gespeichert: italy_sample_tile_aeqd_11.2955_42.9107.tif (389×389 px)


In [ ]:
error

In [ ]:

if __name__ == "__main__":
    positions = positions_ordered
    download_representative_samples(
        positions,
        extent_km=15,
        output_folder="documentation/Representative_Samples",
    )

In [ ]:
"""
Lädt Wasser-Masken (ESA WorldCover 10m v200, Jahr 2021) für die gleichen
Bereiche wie die DEMs aus dem OpenTopography-Skript.

- Quelle: öffentlicher S3-Bucket, kein API-Key, keine Rate Limits
- Auflösung nativ 10m, wird per Average-Resampling auf das DEM-Raster
  gebracht -> weiche Küsten (Anti-Aliasing)
- Ausgabe: 8-bit Graustufen-GeoTIFF (0 = Land, 255 = Wasser, Zwischenwerte
  an Rändern), gleiche Pixelgröße und exakt dasselbe Transform wie das DEM

Voraussetzung: pip install rasterio
"""



# /vsicurl/ lässt GDAL direkt per HTTPS aus dem COG lesen (Range Requests),
# ohne die ganze Kachel herunterzuladen.
WORLDCOVER_URL_TEMPLATE = (
    "/vsicurl/https://esa-worldcover.s3.eu-central-1.amazonaws.com/"
    "v200/2021/map/ESA_WorldCover_10m_2021_v200_{tile}_Map.tif"
)
WATER_CLASS = 80          # WorldCover-Code für "Permanent water bodies"
TILE_SIZE_DEG = 3         # WorldCover-Kacheln sind 3x3 Grad


def _tile_name(lat: int, lon: int) -> str:
    """Kachel-Name aus SW-Ecke. Beispiele: N48E011, S03W015."""
    lat_part = f"N{lat:02d}" if lat >= 0 else f"S{-lat:02d}"
    lon_part = f"E{lon:03d}" if lon >= 0 else f"W{-lon:03d}"
    return f"{lat_part}{lon_part}"


def _tile_urls_for_bbox(west, south, east, north):
    """Alle WorldCover-Kacheln, die die Bounding Box berühren."""
    lat0 = int(np.floor(south / TILE_SIZE_DEG)) * TILE_SIZE_DEG
    lat1 = int(np.floor((north - 1e-9) / TILE_SIZE_DEG)) * TILE_SIZE_DEG
    lon0 = int(np.floor(west / TILE_SIZE_DEG)) * TILE_SIZE_DEG
    lon1 = int(np.floor((east - 1e-9) / TILE_SIZE_DEG)) * TILE_SIZE_DEG

    urls = []
    for lat in range(lat0, lat1 + TILE_SIZE_DEG, TILE_SIZE_DEG):
        for lon in range(lon0, lon1 + TILE_SIZE_DEG, TILE_SIZE_DEG):
            urls.append(WORLDCOVER_URL_TEMPLATE.format(tile=_tile_name(lat, lon)))
    return urls


def download_water_mask(lon, lat, output_folder, label, idx):
    """Erzeugt die Wasser-Maske passend zur bereits vorhandenen DEM-Datei."""
    base       = f"{label}_{idx}_dem_{lon:.4f}_{lat:.4f}"
    dem_path   = os.path.join(output_folder, f"{base}.tif")
    water_path = os.path.join(output_folder, f"{base}_water.tif")

    if not os.path.exists(dem_path):
        print(f"⚠️  DEM fehlt, überspringe: {base}.tif")
        return
    if os.path.exists(water_path):
        print(f"Überspringe {base}_water.tif (existiert bereits)")
        return

    # Ziel-Raster 1:1 vom DEM übernehmen
    with rasterio.open(dem_path) as dem:
        dst_transform = dem.transform
        dst_crs       = dem.crs
        dst_height    = dem.height
        dst_width     = dem.width
        dst_bounds    = dem.bounds

    urls = _tile_urls_for_bbox(
        dst_bounds.left, dst_bounds.bottom, dst_bounds.right, dst_bounds.top
    )
    print(f"Lade {base}_water.tif ({len(urls)} Kachel(n))...")

    srcs = []
    try:
        for url in urls:
            try:
                srcs.append(rasterio.open(url))
            except RasterioIOError:
                # WorldCover liefert keine Kacheln für reine Ozean-Gebiete.
                # Fehlt eine Kachel, wird dieser Bereich später als Wasser gefüllt.
                print(f"  Keine Kachel (Ozean?): {os.path.basename(url)}")

        if not srcs:
            # Komplett außerhalb des Landes -> alles Wasser
            out = np.full((dst_height, dst_width), 255, dtype=np.uint8)
        else:
            # Kleiner Puffer, damit das Resampling keine Randpixel verliert
            pad = 0.002  # ~200 m
            mosaic, src_transform = rio_merge(
                srcs,
                bounds=(
                    dst_bounds.left   - pad,
                    dst_bounds.bottom - pad,
                    dst_bounds.right  + pad,
                    dst_bounds.top    + pad,
                ),
            )

            # Klassenraster -> binäre Maske als float (Wasser = 1.0, sonst 0.0).
            # Durch das Float-Resampling mit "average" entstehen weiche
            # Übergänge an den Küsten; erst ganz am Ende wird auf uint8 gecastet.
            water = (mosaic[0] == WATER_CLASS).astype(np.float32)

            dest = np.zeros((dst_height, dst_width), dtype=np.float32)
            reproject(
                source=water,
                destination=dest,
                src_transform=src_transform,
                src_crs="EPSG:4326",
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=Resampling.average,
            )
            out = np.clip(dest * 255, 0, 255).astype(np.uint8)
    finally:
        for s in srcs:
            s.close()

    profile = {
        "driver":    "GTiff",
        "dtype":     "uint8",
        "count":     1,
        "width":     dst_width,
        "height":    dst_height,
        "crs":       dst_crs,
        "transform": dst_transform,
        "compress":  "deflate",
    }
    with rasterio.open(water_path, "w", **profile) as dst:
        dst.write(out, 1)
    print(f"  Gespeichert: {base}_water.tif")


def download_water_for_positions(positions_dict,
                                 output_folder="Representative_Samples"):
    """Läuft über dieselbe Struktur wie download_representative_samples."""
    os.makedirs(output_folder, exist_ok=True)
    for label, positions in positions_dict.items():
        for idx, position in enumerate(positions):
            lon, lat = float(position[0]), float(position[1])
            download_water_mask(lon, lat, output_folder, label, idx)


if __name__ == "__main__":
    # positions_ordered stammt aus deinem bestehenden Workflow
    positions = positions_ordered  # noqa: F821
    download_water_for_positions(
        positions,
        output_folder="Representative_Samples",
    )